# Running Hugging Face Models Locally with Transformers and Qwen

This notebook demonstrates how to load and run a lightweight, performance-efficient LLM on a local GPU using Hugging Face's `transformers` library.

### Repository
This notebook is hosted at: [ShivaUlavi/GenAI](https://github.com/ShivaUlavi/GenAI)

---

## 🛠️ Installation & Setup
First, we install the required packages. We use a lightweight setup optimized for GPU environments with standard packages like `huggingface_hub`, `transformers`, and `accelerate`.

In [ ]:
#!pip install huggingface_hub

In [ ]:
#!pip install "transformers<4.45.0" "accelerate<0.34.0" --upgrade

## 🔐 Authentication & Model Initialization

To fetch models from Hugging Face Hub (especially private or gated ones), we perform token authentication.

Next, we configure and load our target model: **Qwen2.5-0.5B-Instruct**. This model is chosen for its compact size (~1GB VRAM footprint), making it ideal for standard local GPUs such as the Nvidia GTX 1050 Ti, while utilizing `float16` precision to keep memory overhead to a minimum.

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Authenticate with Hugging Face
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")  # <-- Paste your API key token here
login(token=HF_TOKEN)

# 2. Define a VRAM-friendly model (0.5 Billion parameters ~ 1GB VRAM footprint)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Loading model onto GPU (Active: {torch.cuda.get_device_name(0)})...")
# device_map="cuda" forces the whole model directly onto your 1050 Ti
# torch_dtype=torch.float16 reduces the memory footprint by 50%
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.float16
)



Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\Shivachetan Ulavi\Anaconda3\envs\torch_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shivachetan Ulavi\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model onto GPU (Active: NVIDIA GeForce GTX 1050 Ti)...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 🚀 Local GPU Inference Pipeline

Now, we build a Hugging Face text generation pipeline wrapping our model and tokenizer, format a prompt using the chat template syntax, and prompt the local GPU to generate a response.

In [ ]:
# 3. Create the text generation pipeline
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 4. Format a prompt and test it
messages = [
    {"role": "user", "content": "Give me a short, 2-sentence description of what Quantum Computing is."}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("\n--- Generating Response on GPU ---")
outputs = hf_pipeline(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

# Print the result
generated_text = outputs[0]["generated_text"]
# Strip out the prompt formatting to show just the clean answer
clean_answer = generated_text.split("<|im_start|>assistant\n")[-1].strip()
print(clean_answer)



--- Generating Response on GPU ---


C:\Users\Shivachetan Ulavi\Anaconda3\envs\torch_env\Lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:580: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Quantum computing is a type of computing that utilizes quantum mechanics to process information in ways that classical computers cannot. This allows for the development of new algorithms and solutions that are much faster, more efficient, and potentially capable of solving complex problems that would be impossible for classical computers.


### 🎉 Conclusion
The local pipeline successfully ran using your dedicated GPU to provide fast, efficient inference!